Notebook 1: Damage injection pipeline

This notebook injects synthetic character-level damage at fixed levels (none, 1 word, 10%, 25%, 50%, 75% of a
sentence, around a fact-token anchor). It reads the sample and the OCR-error calibration from Notebook 0
instead of recomputing them, so this notebook only covers turning that calibration into damaged text.

**Inputs.** `data/processed/sentences.jsonl` and `reports/calibration.json`, both written by Notebook 0
(`00_dataset_and_calibration.ipynb`), plus the damage settings in `configs/config.yaml`.

**Outputs.** The frozen damaged dataset `data/processed/corrupted_v2.jsonl`, its hash in `runs/`, and
`reports/corruption_stats.json` (what the simulated damage actually looks like, used by §3.2's process
description and its supporting figures).

Setup

In [1]:
import json

import pandas as pd

from blnrepair.data import ROOT, load_config

pd.set_option("display.max_colwidth", 80)
config = load_config()
print("seed:", config["seed"])

sentences_path = ROOT / config["paths"]["processed_dir"] / "sentences.jsonl"
calibration_path = ROOT / config["paths"]["reports_dir"] / "calibration.json"
assert sentences_path.exists() and calibration_path.exists(), (
    "Run Notebook 0 (00_dataset_and_calibration.ipynb) first -- it writes both input files this notebook reads."
)

seed: 42


Damage Pipeline

For each sentence we pick one **anchor** word among its fact tokens, with a fixed random draw. The damaged words form one block around the anchor, and the block grows with the level: 1 word, then 10%, 25%, 50% and 75% of the words. Level 0 is the clean sentence.

Each word gets one damage plan, made once from a seed built from the sentence id and the word position. A level only decides which words use their plan, so a damaged word looks the same at every level. The plan:

- about 10% of damaged words are dropped completely (our choice, not calibrated), except the anchor word, which is never dropped, so every level damages at least one fact token;
- otherwise 20 to 30% of the word's letters and digits are altered; punctuation is never touched;
- each altered character is deleted (about 15% of the time, as in the calibration, at most 2 per word) or replaced;
- a replacement comes from the calibrated confusion table when that character has entries there (a capital letter uses the entry of its lowercase letter, written as a capital), otherwise from a short list of look-alike characters, otherwise from a random character of the same kind.

In [2]:
from blnrepair.corrupt import select_span, plan_word

pull calibration.json.

In [3]:
import json

from blnrepair.corrupt import settings, load_table, level_sizes

calibration = json.loads((ROOT / config["paths"]["reports_dir"] / "calibration.json").read_text(encoding="utf-8"))
corr = settings(config, calibration)
confusion_table = load_table(calibration)
print(f"Deletion share: {corr['p_deletion']}, intensity: {corr['intensity_low']} to {corr['intensity_high']}, "
      f"word drop: {corr['drop_prob']}, at most {corr['max_deletions']} deletions per word")
print("Damaged words per level for a 25-word sentence:", level_sizes(25))

Deletion share: 0.15, intensity: 0.2 to 0.3, word drop: 0.1, at most 2 deletions per word
Damaged words per level for a 25-word sentence: {'1w': 1, '10': 3, '25': 6, '50': 13, '75': 19}


One test sentence at all six levels. Damaged words are shown in `[[...]]`, a dropped word as `[[]]`.

In [4]:
from blnrepair.corrupt import corrupt_sentence, mark_damage

rows = [json.loads(line) for line in (ROOT / config["paths"]["processed_dir"] / "sentences.jsonl").read_text(encoding="utf-8").splitlines()]
example = rows[0]
levels = [corrupt_sentence(example, level, confusion_table, corr) for level in corr["levels"]]
print(example["id"], "| split:", example["split"], "| words:", example["n_words"], "| anchor:", levels[1]["gold_tokens"][levels[1]["anchor_idx"]])
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame({"level": [r["severity"] for r in levels], "k": [r["k"] for r in levels],
                          "damaged text": [mark_damage(r) for r in levels]}).set_index("level"))

3206211822-001 | split: test | words: 60 | anchor: Joseph


,k,damaged text
level,,
0,0,"On Thursday, Alfred Harris and Joseph Stanfield, a couple of lads in the employ of Messrs. Smith and Co., whip makers, of Beech street, Barbican, were charged, at the Guildhall police-court, with robbing their employers of goods to the value of nearly 100l. , and Joseph Taylor was charged with feloniously receiving the property knowing it to have been stolen."
1w,1,"On Thursday, Alfred Harris and Joseph Stanfield, a couple of lads in the employ of Messrs. Smith and Co., whip makers, of Beech street, Barbican, were charged, at the Guildhall police-court, with robbing their employers of goods to the value of nearly 100l. , and [[Jeseph]] Taylor was charged with feloniously receiving the property knowing it to have been stolen."
10,6,"On Thursday, Alfred Harris and Joseph Stanfield, a couple of lads in the employ of Messrs. Smith and Co., whip makers, of Beech street, Barbican, were charged, at the Guildhall police-court, with robbing their employers of goods to the value of nearly [[10l.]] [[]] [[nd]] [[Jeseph]] [[Tey1or]] [[wag]] charged with feloniously receiving the property knowing it to have been stolen."
25,15,"On Thursday, Alfred Harris and Joseph Stanfield, a couple of lads in the employ of Messrs. Smith and Co., whip makers, of Beech street, Barbican, were charged, at the Guildhall police-court, with robbing their employers of goods to [[bhe]] [[valne]] [[ot]] [[uear1y]] [[10l.]] [[]] [[nd]] [[Jeseph]] [[Tey1or]] [[wag]] [[clarge]] [[vith]] [[f1onieusly]] [[iecelving]] [[bhe]] property knowing it to have been stolen."
50,30,"On Thursday, Alfred Harris and Joseph Stanfield, a couple of lads in the employ of Messrs. Smith and Co., whip makers, of Beech street, Barbican, were charged, at the Guildhall [[poliee-ceurt,]] [[wth]] [[iobbIng]] [[]] [[empoyere]] [[]] [[goeds]] [[]] [[bhe]] [[valne]] [[ot]] [[uear1y]] [[10l.]] [[]] [[nd]] [[Jeseph]] [[Tey1or]] [[wag]] [[clarge]] [[vith]] [[f1onieusly]] [[iecelving]] [[bhe]] [[prepsrty]] [[koWing]] [[lt]] [[te]] [[hane]] [[beei]] [[ebolen.]]"
75,45,"On Thursday, Alfred Harris and Joseph Stanfield, a couple of lads in the employ of [[Mesars.]] [[]] [[]] [[C.,]] [[whii]] [[mekers,]] [[ot]] [[Beoch]] [[sbreet,]] [[Harbioan,]] [[weie]] [[hsrged,]] [[st]] [[tbe]] [[Guilcliall]] [[poliee-ceurt,]] [[wth]] [[iobbIng]] [[]] [[empoyere]] [[]] [[goeds]] [[]] [[bhe]] [[valne]] [[ot]] [[uear1y]] [[10l.]] [[]] [[nd]] [[Jeseph]] [[Tey1or]] [[wag]] [[clarge]] [[vith]] [[f1onieusly]] [[iecelving]] [[bhe]] [[prepsrty]] [[koWing]] [[lt]] [[te]] [[hane]] [[beei]] [[ebolen.]]"


What the damage looks like over all 170 sentences. We use level 75, which holds every damaged word, because smaller blocks lie inside it. *Intensity* is the share of a word's letters and digits that were altered. The second table gives the character error rate of the whole damaged sentence against the gold sentence at each level, which can be compared with the real OCR error rate (about 0.07 for a whole excerpt).

In [5]:
from blnrepair.corrupt import corruption_stats, cer_by_level

stats = corruption_stats([corrupt_sentence(r, "75", confusion_table, corr) for r in rows])
stats["cer_by_level"] = cer_by_level(rows, confusion_table, corr)
(ROOT / config["paths"]["reports_dir"] / "corruption_stats.json").write_text(json.dumps(stats, indent=1), encoding="utf-8")
display(pd.json_normalize({k: v for k, v in stats.items() if k != "cer_by_level"}).T.rename(columns={0: "value"}).round(3))
print("Character error rate of the damaged sentence against the gold sentence, by level (170 sentences):")
pd.DataFrame(stats["cer_by_level"]).T.rename_axis("level").round(3)

,value
damaged_words,4263.000
share_dropped,0.099
deletion_share_of_altered_chars,0.146
digit_words,87.000
no_alnum_tokens,7.000
substitution_source_share.table,0.857
substitution_source_share.table_casefold,0.036
substitution_source_share.fallback_map,0.049
substitution_source_share.fallback_random,0.058
mean_intensity_by_length.1-3,0.451


Character error rate of the damaged sentence against the gold sentence, by level (170 sentences):


,mean,median,q1,q3
level,,,,
0,0.000,0.000,0.000,0.000
1w,0.010,0.009,0.006,0.013
10,0.032,0.027,0.021,0.038
25,0.079,0.073,0.063,0.091
50,0.155,0.149,0.131,0.174
75,0.231,0.226,0.202,0.257


In [6]:
from blnrepair.corrupt import review_examples, review_table

review = review_table(review_examples(rows, confusion_table, corr))
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(review)

,level,band,k,fact words in span,gold sentence,damaged (marked)
id,,,,,,
3200811564-027,1w,20-29,1,Sutton.,"He said that, as arranged on Good Friday, he sent her his address, but he could not see that this enticed her from Sutton.","He said that, as arranged on Good Friday, he sent her his address, but he could not see that this enticed her from [[Suttan.]]"
3200812378-002,1w,30-44,1,Baroness,"The Baroness de Rahden, the heroine of the drama, was a circus rider of the haute ecole, Lieutenant de Kastenkjold made himself conspicuous by his admiration of the baroness, who allowed him to follow her from town to town.","The [[Baonese]] de Rahden, the heroine of the drama, was a circus rider of the haute ecole, Lieutenant de Kastenkjold made himself conspicuous by his admiration of the baroness, who allowed him to follow her from town to town."
3206222380-003,1w,45-60,1,Jabez,"She had been living in the service of the Rev. Jabez Tunnicliffe, who is the officiating minister at the Burmantofts cemetery, Leeds, but left suddenly on Sunday night week, and went to reside with Mrs. Waite, of Napier-street, Burmantofts, who is the married daughter of Mr. Tunnicliffe.","She had been living in the service of the Rev. [[Jahez]] Tunnicliffe, who is the officiating minister at the Burmantofts cemetery, Leeds, but left suddenly on Sunday night week, and went to reside with Mrs. Waite, of Napier-street, Burmantofts, who is the married daughter of Mr. Tunnicliffe."
3206327837-020,1w,20-29,1,Lawrence,"This was an indictment for conspiracy, the prosecutor being Mr. Lawrence Levy, formerly an attorney, and the defendants being a firm of solicitors.","This was an indictment for conspiracy, the prosecutor being Mr. [[Lawenc]] Levy, formerly an attorney, and the defendants being a firm of solicitors."
3206329007-012,10,30-44,3,Wright,"He also stated he had seen Wright put three nuts into the cog wheel on three consecutive mornings, while Wright on the other hand accused him of being the guilty party.","He also stated he had seen Wright put three nuts into the cog wheel on three consecutive mornings, [[whlle]] [[Viight]] [[en]] the other hand accused him of being the guilty party."
3206203053-011,10,45-60,5,"Maria, Froggatt","Anna Maria Froggatt deposed that she first made prisoner's acquaintance at Preston, in 1847, when he induced her to leave her home, and accompany him to Gretna-hall, in Scotland, where they were married, according to the ceremony observed there, and pronounced by the Rev. Mr. Linton, the priest, to be legally man and wife.","[[Snna]] [[Naria]] [[Froggst]] [[clepoed]] [[tbat]] she first made prisoner's acquaintance at Preston, in 1847, when he induced her to leave her home, and accompany him to Gretna-hall, in Scotland, where they were married, according to the ceremony observed there, and pronounced by the Rev. Mr. Linton, the priest, to be legally man and wife."
3206232665-017,10,20-29,2,Wallace,"The impression was that the child had been taken away by the Wallace’s, Mrs. Wallace being a very strange woman.","The impression was that the child had been taken away by the Wallace’s, [[Mss.]] [[Wellsce]] being a very strange woman."
3200810696-001,10,30-44,4,"Thomas, Hurd,","A POWERFULLY-BUILT man, named Thomas Hurd, a commission agent, who refused his address, was charged, at Marylebone Police-court, on Saturday, with being concerned, with others, in obtaining 10s, by means of a trick from Frank Bliss, a medical student, of 29, Paddington-green.","A POWERFULLY-BUILT man, [[iamed]] [[Bhomas]] [[Husd,]] [[e]] commission agent, who refused his address, was charged, at Marylebone Police-court, on Saturday, with being concerned, with others, in obtaining 10s, by means of a trick from Frank Bliss, a medical student, of 29, Paddington-green."
3206223815-014,25,45-60,14,"Police-constable, Abbott","He had seen a quantity of sovereigns in the possession of Police-constable Abbott that morning, and he could swear that some of them were his prope

Freeze

Damaged dataset is written once and frozen, so that both repair methods later see exactly the same inputs. `data/processed/corrupted_v2.jsonl` has one row per sentence and level (170 sentences x 6 levels = 1,020 rows, in the order of `sentences.jsonl`, then level 0, 1w, 10, 25, 50, 75). 

`runs/corrupted_v2.sha256` holds the hash of the file
`runs/config_snapshot_v2.yaml` holds the version, seed, all damage settings, the hashes of the two input files and the package versions. 

In [7]:
from blnrepair.freeze import build_records, freeze, verify_freeze

records = build_records(rows, confusion_table, corr)
processed_dir, reports_dir, runs_dir = (ROOT / config["paths"][k] for k in ("processed_dir", "reports_dir", "runs_dir"))
status, digest = freeze(records, "v2", corr, processed_dir, reports_dir, runs_dir)
print(f"corrupted_v2.jsonl: {status}")

corrupted_v2.jsonl: unchanged


In [8]:
rebuilt = verify_freeze(rows, confusion_table, corr, "v2", processed_dir, runs_dir)
size = (processed_dir / "corrupted_v2.jsonl").stat().st_size
print(f"sha256: {rebuilt}")
print("The rebuilt hash equals the stored hash and the file on disk.")
print(f"Rows: {len(records):,}, file size: {size:,} bytes")
pd.crosstab(pd.Series([r["severity"] for r in records], name="level"),
            pd.Series([r["split"] for r in records], name="split"), margins=True).reindex(corr["levels"] + ["All"])

sha256: 5798336d99ff2a8ada306e0752bcd25298287d0bf9f0ad4b5fc0c6468a6b9eb9
The rebuilt hash equals the stored hash and the file on disk.
Rows: 1,020, file size: 2,319,195 bytes


split,dev,test,All
level,,,
0,20,150,170
1w,20,150,170
10,20,150,170
25,20,150,170
50,20,150,170
75,20,150,170
All,120,900,1020


## Numbers used in the report (this notebook)

Recomputed from `stats` (this notebook's `corruption_stats.json`) rather than retyped.

In [9]:
report_numbers_nb1 = []

report_numbers_nb1.append(("\u00a73.2 Damage Simulation", "word drop probability (config, not calibrated)",
    f"{corr['drop_prob']:.0%}"))

report_numbers_nb1.append(("\u00a73.2 Damage Simulation / noise-process figure",
    "realized share of damaged words dropped (simulation, level 75)",
    f"{stats['share_dropped']:.1%}"))

report_numbers_nb1.append(("\u00a73.2 Damage Simulation / noise-process figure",
    "of a word's altered characters: deleted vs. substituted",
    f"{stats['deletion_share_of_altered_chars']:.1%} deleted, "
    f"{1 - stats['deletion_share_of_altered_chars']:.1%} substituted"))

src_share = stats["substitution_source_share"]
report_numbers_nb1.append(("\u00a73.2 Damage Simulation / noise-process figure",
    "substitution source shares",
    f"table {src_share['table']:.1%}, capitalised {src_share['table_casefold']:.1%}, "
    f"look-alike {src_share['fallback_map']:.1%}, random {src_share['fallback_random']:.1%}"))

mi = stats["mean_intensity_by_length"]
report_numbers_nb1.append(("\u00a73.2 Damage Simulation, short-word caveat",
    "mean realized intensity by word length (1–3 / 4–6 / 7+ letters)",
    f"{mi['1-3']:.1%} / {mi['4-6']:.1%} / {mi['7+']:.1%} (nominal target: 20–30%)"))

cer = stats["cer_by_level"]
report_numbers_nb1.append(("Table: damage by level", "sentence CER by level (mean)",
    ", ".join(f"{lvl}: {v['mean']:.1%}" for lvl, v in cer.items())))

report_numbers_nb1.append(("Frozen dataset", "rows written to corrupted_v2.jsonl",
    f"{len(records):,} ({len(rows)} sentences x {len(corr['levels'])} levels), sha256 {digest[:12]}..."))

recap_table_nb1 = pd.DataFrame(report_numbers_nb1, columns=["Report location", "Quantity", "Value (recomputed here)"])
with pd.option_context("display.max_colwidth", 200):
    display(recap_table_nb1)
recap_table_nb1.to_csv(ROOT / config["paths"]["reports_dir"] / "report_numbers_notebook1.csv", index=False)

,Report location,Quantity,Value (recomputed here)
0,§3.2 Damage Simulation,"word drop probability (config, not calibrated)",10%
1,§3.2 Damage Simulation / noise-process figure,"realized share of damaged words dropped (simulation, level 75)",9.9%
2,§3.2 Damage Simulation / noise-process figure,of a word's altered characters: deleted vs. substituted,"14.6% deleted, 85.4% substituted"
3,§3.2 Damage Simulation / noise-process figure,substitution source shares,"table 85.7%, capitalised 3.6%, look-alike 4.9%, random 5.8%"
4,"§3.2 Damage Simulation, short-word caveat",mean realized intensity by word length (1–3 / 4–6 / 7+ letters),45.1% / 23.4% / 25.6% (nominal target: 20–30%)
5,Table: damage by level,sentence CER by level (mean),"0: 0.0%, 1w: 1.0%, 10: 3.2%, 25: 7.9%, 50: 15.5%, 75: 23.1%"
6,Frozen dataset,rows written to corrupted_v2.jsonl,"1,020 (170 sentences x 6 levels), sha256 5798336d99ff..."
